# Lesion-Scale-Aware Boundary Preservation under Speckle Corruption
### UltraLight VM-UNet (~0.049M Params) on TN3K Thyroid Ultrasound Dataset

**Speed-Optimized Architecture:**
- Uses **JIT-compiled accelerated selective scan** and **GPU-vectorized mixed-precision (AMP)**.
- Eliminates slow 1-2 hour CUDA kernel builds—everything starts running in seconds!
- Automatically saves best checkpoints to your Google Drive.

### 1. Hardware & GPU Check

In [ ]:
!nvidia-smi

### 2. Fast Dependency Installation (Under 20 Seconds)
Installs all necessary packages without any hanging C++/CUDA compilation.

In [ ]:
!pip install -q pyyaml scipy timm thop packaging ninja
print("✅ Core dependencies installed in seconds!")

### 3. Clone or Update Project Code from GitHub

In [ ]:
import os

# Set your repository URL (Replace with your actual GitHub repo)
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/ML_UVnet.git"

if not os.path.exists('/content/ML_UVnet'):
    !git clone $REPO_URL /content/ML_UVnet
else:
    %cd /content/ML_UVnet
    !git pull

%cd /content/ML_UVnet

# Instant verification: Test UltraLight VM-UNet forward pass (runs in 1 second)
import torch
from models import UltraLight_VM_UNet
test_model = UltraLight_VM_UNet(num_classes=1).cuda()
dummy = torch.randn(1, 3, 256, 256).cuda()
with torch.no_grad():
    out = test_model(dummy)
print("\n✅ SUCCESS: UltraLight VM-UNet initialized and verified!")
print(f"Output Shape: {out.shape} | Parameters: {sum(p.numel() for p in test_model.parameters()) / 1e6:.4f}M")

### 4. Mount Google Drive and Unpack TN3K Dataset Locally
Transfers `TN3K.zip` to Colab NVMe scratch disk and extracts it in under 15 seconds.

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

drive_zip = '/content/drive/MyDrive/TN3K.zip'
drive_folder = '/content/drive/MyDrive/TN3K_extracted/Thyroid Dataset/tn3k'
local_dir = '/content/tn3k'

if not os.path.exists(local_dir):
    if os.path.exists(drive_zip):
        print("Found TN3K.zip! Copying to local /content/...")
        shutil.copy(drive_zip, '/content/TN3K.zip')
        print("Unzipping to /content/tn3k...")
        !unzip -q /content/TN3K.zip -d /content/
        if not os.path.exists(local_dir) and os.path.exists('/content/Thyroid Dataset/tn3k'):
            !mv "/content/Thyroid Dataset/tn3k" /content/tn3k
        elif not os.path.exists(local_dir) and os.path.exists('/content/TN3K'):
            !mv /content/TN3K /content/tn3k
    elif os.path.exists(drive_folder):
        print("Copying extracted folder from Drive...")
        !cp -r "$drive_folder" /content/tn3k

print("\n--- Dataset Verification ---")
print("TrainVal Images:", len(os.listdir('/content/tn3k/trainval-image')))
print("Test Images:    ", len(os.listdir('/content/tn3k/test-image')))
print("Folds:          ", [f for f in os.listdir('/content/tn3k') if f.endswith('.json')])
print("Dataset ready at /content/tn3k")

### 5. Experiment 1 (E1): UltraLight VM-UNet Baseline Reproduction
Tip: Set `--epochs 50` for a fast initial run (~5-8 mins), or `--epochs 150` for full benchmark convergence.

In [ ]:
%cd /content/ML_UVnet
!python train.py --config configs/e1_baseline.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e1_baseline

### 6. Experiment 2 (E2): Lesion-Scale-Aware Boundary Supervision

In [ ]:
%cd /content/ML_UVnet
!python train.py --config configs/e2_boundary.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e2_boundary

### 7. Experiment 3 (E3): Corruption-Aware Consistency Training

In [ ]:
%cd /content/ML_UVnet
!python train.py --config configs/e3_consistency.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e3_consistency

### 8. Experiment 4 (E4): Full Combined Proposed Method

In [ ]:
%cd /content/ML_UVnet
!python train.py --config configs/e4_combined.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e4_combined

### 9. Final Benchmark Evaluation, Size Breakdown & Robustness Curves
Evaluates on the held-out test set: Dice, IoU, Boundary IoU, HD95, latency, FPS, and speckle noise degradation curve.

In [ ]:
%cd /content/ML_UVnet
!python evaluate.py --weights /content/drive/MyDrive/ML_UVnet_checkpoints/e4_combined/best_model.pth \
                    --data_root /content/tn3k \
                    --output_dir /content/drive/MyDrive/ML_UVnet_checkpoints/eval_e4

### 10. Display Qualitative Visual Overlay Segmentations

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

eval_vis_dir = '/content/drive/MyDrive/ML_UVnet_checkpoints/eval_e4/visual_predictions'
images = sorted(glob.glob(f"{eval_vis_dir}/*.png"))[:4]

if images:
    fig, axes = plt.subplots(len(images), 1, figsize=(14, 4 * len(images)))
    if len(images) == 1:
        axes = [axes]
    for ax, img_p in zip(axes, images):
        ax.imshow(Image.open(img_p))
        ax.set_title("[Input Ultrasound Image  |  Ground Truth Mask  |  Model Prediction]", fontsize=13)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No visual prediction PNGs found. Run evaluate.py first!")